# 🧪 Agent Chat + Validation Workflow Tests

Tests the new conversational chat node, validation framework, and extended use case catalog.

## Test Cases
1. Basic site scoring via chat
2. Follow-up question
3. Off-topic handling
4. Missing field extraction
5. Validation blocking (liquor store in Gujarat)
6. Validation warning (hospital in flood zone)
7. Direct answer
8. Hotspot via chat
9. Extended use case catalog
10. Graph compilation check

In [1]:
import sys
sys.path.insert(0, 'E:/geo_site_v2')

import asyncio
import uuid
from pprint import pprint

## Test 1: Use Case Catalog

In [2]:
from scoring.weights import (
    USE_CASE_CATALOG, VALID_USE_CASES, USE_CASE_CATEGORIES,
    USE_CASE_WEIGHTS, get_default_weights, get_use_case_by_category,
    AGRI_USE_CASES,
)

print(f"Total use cases: {len(USE_CASE_CATALOG)}")
print(f"Total categories: {len(USE_CASE_CATEGORIES)}")
print(f"VALID_USE_CASES count: {len(VALID_USE_CASES)}")
print(f"Backward-compat USE_CASE_WEIGHTS count: {len(USE_CASE_WEIGHTS)}")
print(f"Agri use cases: {AGRI_USE_CASES}")
print()

# List by category
for cat_key, cat_name in USE_CASE_CATEGORIES.items():
    entries = get_use_case_by_category(cat_key)
    names = [e.name for e in entries]
    print(f"{cat_name} ({len(entries)}): {', '.join(names[:5])}{'...' if len(names) > 5 else ''}")

Total use cases: 95
Total categories: 15
VALID_USE_CASES count: 95
Backward-compat USE_CASE_WEIGHTS count: 95
Agri use cases: {'agri_cold_chain', 'greenhouse', 'aquaculture', 'agri_input_store', 'agri_mandi'}

Necessity & Civic (8): Public Toilet / Sanitation, Water Supply Point, Post Office, Police Station, Fire Station...
Healthcare (10): Hospital, Clinic, Diagnostic Lab, Pharmacy, Pharmacy Chain Outlet...
Education (6): School, College / University, Skill Development Centre, Anganwadi / Child Care, Public Library...
Food & Beverage (11): Restaurant, Café, Bakery, Cloud Kitchen, Food Processing Unit...
Retail & Commerce (5): Retail Store, Supermarket, Auto / Electronics Showroom, Auto Service / Garage, Tobacco / Pan Shop
Energy & Fuel (9): Petrol Pump, CNG Station, EV Charging Station, EV Battery Swap Station, Solar Power Plant...
Finance & Banking (4): Bank Branch, ATM Installation, Microfinance Office, Insurance Office
Fun & Leisure (9): Multiplex / Cinema, Amusement Park, Gaming Z

In [3]:
# Verify all weights sum to 1.0
errors = []
for key, cfg in USE_CASE_CATALOG.items():
    w = cfg.weights
    total = (w.demand_score + w.accessibility_score + w.competition_score +
             w.suitability_score + w.risk_score + w.infrastructure_score)
    if not (0.99 <= total <= 1.01):
        errors.append(f"{key}: sum={total:.3f}")

if errors:
    print(f"Weight errors: {errors}")
else:
    print(f"✅ All {len(USE_CASE_CATALOG)} use cases have valid weights (sum=1.0)")

✅ All 95 use cases have valid weights (sum=1.0)


In [4]:
# Test backward compatibility
for old_key in ['retail', 'ev_charging', 'warehouse', 'telecom', 'renewable']:
    w = get_default_weights(old_key)
    print(f"{old_key}: demand={w.demand_score}, access={w.accessibility_score}")

print("\n✅ Old 5-key system still works")

retail: demand=0.3, access=0.2
ev_charging: demand=0.25, access=0.3
warehouse: demand=0.15, access=0.35
telecom: demand=0.1, access=0.25
renewable: demand=0.1, access=0.2

✅ Old 5-key system still works


## Test 2: Validation Framework

In [5]:
from tools.validation_tools import (
    validate_legal, validate_proximity, validate_environmental,
    validate_land_use, validate_viability, run_all_validations,
    ValidationResult,
)

# Mock site features
class MockFeatures:
    def __init__(self, **kwargs):
        defaults = {
            'school_count': 0, 'hospital_count': 0,
            'residential_ratio': 0.20, 'industrial_ratio': 0.10,
            'commercial_ratio': 0.30, 'mixed_use_ratio': 0.20,
            'aqi': 80, 'flood_risk_score': 30, 'earthquake_risk_score': 20,
            'population_density': 500, 'competitor_count': 5,
            'electricity_access_score': 80, 'accessibility_score': 70,
            'built_up_area_ratio': 0.40, 'water_body_proximity': 500,
        }
        defaults.update(kwargs)
        for k, v in defaults.items():
            setattr(self, k, v)

print("Mock features created")

Mock features created


In [6]:
# Test 2a: Legal — liquor store in Gujarat (should BLOCK)
r = validate_legal('liquor_store', 'Gujarat')
print(f"Liquor in Gujarat: {r.status} — {r.reason}")
assert r.status == 'block', f"Expected block, got {r.status}"
print("✅ PASS")

# Test 2b: Legal — retail in Gujarat (should PASS)
r = validate_legal('retail', 'Gujarat')
print(f"Retail in Gujarat: {r.status}")
assert r.status == 'pass'
print("✅ PASS")

# Test 2c: Legal — plastic factory anywhere (should BLOCK)
r = validate_legal('plastic_factory', 'Maharashtra')
print(f"Plastic factory in Maharashtra: {r.status} — {r.reason}")
assert r.status == 'block'
print("✅ PASS")

Liquor in Gujarat: block — Alcohol is prohibited in Gujarat under state excise law.
✅ PASS
Retail in Gujarat: pass
✅ PASS
Plastic factory in Maharashtra: block — Single-use plastic manufacturing is banned nationwide.
✅ PASS


In [7]:
# Test 2d: Proximity — tobacco near school (should BLOCK)
f = MockFeatures(school_count=2)
r = validate_proximity('tobacco_shop', f)
print(f"Tobacco near school: {r.status} — {r.reason}")
assert r.status == 'block'
print("✅ PASS")

# Test 2e: Proximity — chemical in residential (should BLOCK)
f = MockFeatures(residential_ratio=0.45)
r = validate_proximity('chemical_factory', f)
print(f"Chemical in residential: {r.status} — {r.reason}")
assert r.status == 'block'
print("✅ PASS")

# Test 2f: Proximity — hospital near industrial (should WARN)
f = MockFeatures(industrial_ratio=0.35)
r = validate_proximity('hospital', f)
print(f"Hospital near industrial: {r.status} — {r.reason} (penalty: {r.penalty_dimension} -{r.penalty_points})")
assert r.status == 'warn'
print("✅ PASS")

Tobacco near school: block — Tobacco shops cannot operate within 100 m of any educational institution.
✅ PASS
Chemical in residential: block — Hazardous industrial units cannot be sited in residential zones (>30 % residential).
✅ PASS
Hospital near industrial: warn — Healthcare facilities near heavy industrial zones face AQI and noise compliance issues. (penalty: risk_score -15)
✅ PASS


In [8]:
# Test 2g: Environmental — school in flood zone (should BLOCK)
f = MockFeatures(flood_risk_score=90)
r = validate_environmental('school', f)
print(f"School in flood zone: {r.status} — {r.reason}")
assert r.status == 'block'
print("✅ PASS")

# Test 2h: Land use — retail on agricultural land (should BLOCK)
f = MockFeatures(commercial_ratio=0.01, residential_ratio=0.01, industrial_ratio=0.01)
r = validate_land_use('retail', f)
print(f"Retail on agri land: {r.status} — {r.reason}")
assert r.status == 'block'
print("✅ PASS")

# Test 2i: Land use — greenhouse on agri land (should PASS)
r = validate_land_use('greenhouse', f)
print(f"Greenhouse on agri land: {r.status}")
assert r.status == 'pass'
print("✅ PASS")

School in flood zone: block — Critical infrastructure cannot be sited in high flood risk zones (score > 80).
✅ PASS
Retail on agri land: block — This appears to be agricultural land. Non-Agricultural (NA) land conversion is required before any commercial or industrial development.
✅ PASS
Greenhouse on agri land: pass
✅ PASS


In [9]:
# Test 2j: Viability — salon in low density area (should WARN)
f = MockFeatures(population_density=30)
results = validate_viability('salon', f)
print(f"Salon low density: {len(results)} warnings")
for r in results:
    print(f"  {r.status}: {r.reason} (penalty: {r.penalty_dimension} -{r.penalty_points})")
assert len(results) >= 1
print("✅ PASS")

Salon low density: 1 warnings
  warn: Very low population density (<50/km²). Walk-in footfall may be insufficient for viability. (penalty: demand_score -20)
✅ PASS


In [10]:
# Test 2k: run_all_validations — liquor store in Gujarat
f = MockFeatures(school_count=1)
results = run_all_validations('liquor_store', 'Gujarat', f)
print(f"Total results: {len(results)}")
for r in results:
    print(f"  [{r.status.upper()}] {r.reason}")

# Should have at least the legal block
blocks = [r for r in results if r.status == 'block']
assert len(blocks) >= 1, "Expected at least one block"
print("\n✅ run_all_validations PASS")

Total results: 2
  [BLOCK] Alcohol is prohibited in Gujarat under state excise law.
  [BLOCK] Liquor shops cannot be within 500 m of schools or hospitals.

✅ run_all_validations PASS


## Test 3: Graph Compilation

In [11]:
from agents.graph import build_graph, get_compiled_graph

graph = build_graph()
compiled = graph.compile()

nodes = list(compiled.get_graph().nodes.keys())
print(f"Graph nodes ({len(nodes)}):")
for n in nodes:
    print(f"  • {n}")

# Verify expected nodes exist
expected = ['chat', 'orchestrator', 'advisory', 'fetch_features',
            'fetch_scores', 'validation', 'compute_score', 'geospatial',
            'explainability', 'insight', 'chat_response', 'error_handler']
for e in expected:
    assert e in nodes, f"Missing node: {e}"

print(f"\n✅ All {len(expected)} expected nodes present")

Graph nodes (14):
  • __start__
  • chat
  • orchestrator
  • advisory
  • fetch_features
  • fetch_scores
  • validation
  • compute_score
  • geospatial
  • explainability
  • insight
  • chat_response
  • error_handler
  • __end__

✅ All 12 expected nodes present


## Test 4: Chat Node — Intent Detection

In [12]:
from agents.chat import chat_node, _normalize_use_case

# Test use case normalization
test_cases = [
    ('retail', 'retail'),
    ('EV Charging', 'ev_charging'),
    ('ev charging station', 'ev_charging'),
    ('hospital', 'hospital'),
    ('petrol pump', 'petrol_pump'),
    ('gym', 'gym'),
    ('salon', 'salon'),
    ('warehouse', 'warehouse'),
    ('bank', 'bank_branch'),
    ('school', 'school'),
    ('hotel', 'hotel'),
    ('solar', 'solar_plant'),
]

for raw, expected in test_cases:
    result = _normalize_use_case(raw)
    status = '✅' if result == expected else '❌'
    print(f"  {status} '{raw}' → '{result}' (expected: '{expected}')")

print("\nNormalization tests complete")

  ✅ 'retail' → 'retail' (expected: 'retail')
  ✅ 'EV Charging' → 'ev_charging' (expected: 'ev_charging')
  ✅ 'ev charging station' → 'ev_charging' (expected: 'ev_charging')
  ✅ 'hospital' → 'hospital' (expected: 'hospital')
  ✅ 'petrol pump' → 'petrol_pump' (expected: 'petrol_pump')
  ✅ 'gym' → 'gym' (expected: 'gym')
  ✅ 'salon' → 'salon' (expected: 'salon')
  ✅ 'warehouse' → 'warehouse' (expected: 'warehouse')
  ✅ 'bank' → 'bank_branch' (expected: 'bank_branch')
  ✅ 'school' → 'school' (expected: 'school')
  ✅ 'hotel' → 'hotel' (expected: 'hotel')
  ✅ 'solar' → 'solar_plant' (expected: 'solar_plant')

Normalization tests complete


## Test 5: Chat Node — Off-Topic Handling

In [13]:
from agents.chat import _handle_off_topic

# Test escalating responses
for i in range(3):
    response = _handle_off_topic(i)
    print(f"Retry {i}: {response[:80]}...")

print("\n✅ Off-topic escalation works")

Retry 0: I can only help with site selection and business location analysis. Try asking: ...
Retry 1: Let's stay focused on location analysis. What business type and location would y...
Retry 2: I'm designed specifically for geospatial site analysis. This session will end af...

✅ Off-topic escalation works


## Test 6: Full Graph Run — Score a Site

⚠️ Requires database connection and LLM API key.

In [14]:
async def test_full_score():
    compiled = get_compiled_graph()
    state = {
        'raw_user_message': 'Score a retail site at 23.02, 72.57',
        'conversation_history': [],
        'retry_count': 0,
        'analysis_complete': False,
        'thread_id': str(uuid.uuid4()),
    }
    config = {'configurable': {'thread_id': state['thread_id']}}
    result = await compiled.ainvoke(state, config=config)
    
    print(f"Chat intent: {result.get('chat_intent')}")
    print(f"Graph intent: {result.get('intent')}")
    print(f"Final score: {result.get('final_score')}")
    print(f"Chat response: {result.get('chat_response', '')[:200]}")
    print(f"Validation warnings: {result.get('validation_warnings', [])}")
    print(f"Analysis complete: {result.get('analysis_complete')}")
    return result

result = await test_full_score()

2026-04-12 03:50:18,789 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-04-12 03:50:18,789 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-12 03:50:18,791 INFO sqlalchemy.engine.Engine select current_schema()
2026-04-12 03:50:18,791 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-12 03:50:18,791 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-04-12 03:50:18,791 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-12 03:50:18,795 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-12 03:50:18,795 INFO sqlalchemy.engine.Engine 
        SELECT *
        FROM site_features
        ORDER BY geom <-> ST_SetSRID(ST_MakePoint($1, $2), 4326)
        LIMIT 1;
    
2026-04-12 03:50:18,795 INFO sqlalchemy.engine.Engine [generated in 0.00069s] (72.57, 23.02)
2026-04-12 03:50:18,824 INFO sqlalchemy.engine.Engine ROLLBACK
2026-04-12 03:50:18,827 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-12 03:50:18,827 INFO sqlalchemy.engine.Engine 
        SELEC

## Test 7: Full Graph Run — Validation Blocking

⚠️ Requires database connection and LLM API key.

In [15]:
async def test_validation_block():
    compiled = get_compiled_graph()
    state = {
        'raw_user_message': 'Score a liquor store at 23.02, 72.57 in Gujarat',
        'conversation_history': [],
        'retry_count': 0,
        'analysis_complete': False,
        'thread_id': str(uuid.uuid4()),
    }
    config = {'configurable': {'thread_id': state['thread_id']}}
    result = await compiled.ainvoke(state, config=config)
    
    print(f"Error (expected block): {result.get('error', 'None')[:200]}")
    print(f"Chat response: {result.get('chat_response', '')[:200]}")
    assert result.get('error'), 'Expected validation block error'
    print('\n✅ Validation blocking works')
    return result

result = await test_validation_block()

2026-04-12 03:50:21,316 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-12 03:50:21,318 INFO sqlalchemy.engine.Engine 
        SELECT *
        FROM site_features
        ORDER BY geom <-> ST_SetSRID(ST_MakePoint($1, $2), 4326)
        LIMIT 1;
    
2026-04-12 03:50:21,318 INFO sqlalchemy.engine.Engine [cached since 2.522s ago] (72.57, 23.02)
2026-04-12 03:50:21,320 INFO sqlalchemy.engine.Engine ROLLBACK
2026-04-12 03:50:21,327 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-12 03:50:21,327 INFO sqlalchemy.engine.Engine 
        SELECT id,
               demand_score,
               accessibility_score,
               competition_score,
               suitability_score,
               risk_score,
               infrastructure_score
        FROM site_features
        WHERE id = $1
        LIMIT 1
        
2026-04-12 03:50:21,327 INFO sqlalchemy.engine.Engine [cached since 2.499s ago] ('IND_0099352',)
2026-04-12 03:50:21,327 INFO sqlalchemy.engine.Engine ROLLBACK
Error (exp

## Test 8: Follow-Up After Analysis

⚠️ Requires database connection and LLM API key.

In [16]:
async def test_follow_up():
    compiled = get_compiled_graph()
    
    # First: score a site
    state = {
        'raw_user_message': 'Score a retail site at 23.02, 72.57',
        'conversation_history': [],
        'retry_count': 0,
        'analysis_complete': False,
        'thread_id': str(uuid.uuid4()),
    }
    config = {'configurable': {'thread_id': state['thread_id']}}
    result1 = await compiled.ainvoke(state, config=config)
    print(f"Score: {result1.get('final_score')}")
    
    # Second: follow-up question
    state2 = {
        'raw_user_message': 'Why is the risk score low?',
        'conversation_history': result1.get('conversation_history', []),
        'retry_count': 0,
        'analysis_complete': True,
        'thread_id': state['thread_id'],
        'site_features': result1.get('site_features'),
        'score_breakdown': result1.get('score_breakdown'),
        'final_score': result1.get('final_score'),
        'use_case': result1.get('use_case'),
    }
    result2 = await compiled.ainvoke(state2, config=config)
    print(f"\nFollow-up intent: {result2.get('chat_intent')}")
    print(f"Follow-up response: {result2.get('chat_response', '')[:300]}")
    return result2

result = await test_follow_up()

2026-04-12 03:50:23,719 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-12 03:50:23,721 INFO sqlalchemy.engine.Engine 
        SELECT *
        FROM site_features
        ORDER BY geom <-> ST_SetSRID(ST_MakePoint($1, $2), 4326)
        LIMIT 1;
    
2026-04-12 03:50:23,722 INFO sqlalchemy.engine.Engine [cached since 4.925s ago] (72.57, 23.02)
2026-04-12 03:50:23,726 INFO sqlalchemy.engine.Engine ROLLBACK
2026-04-12 03:50:23,734 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-12 03:50:23,736 INFO sqlalchemy.engine.Engine 
        SELECT id,
               demand_score,
               accessibility_score,
               competition_score,
               suitability_score,
               risk_score,
               infrastructure_score
        FROM site_features
        WHERE id = $1
        LIMIT 1
        
2026-04-12 03:50:23,738 INFO sqlalchemy.engine.Engine [cached since 4.909s ago] ('IND_0099352',)
2026-04-12 03:50:23,741 INFO sqlalchemy.engine.Engine ROLLBACK
Score: 69.

## Test 9: Config Tools Backward Compat

In [17]:
from tools.config_tools import get_default_weights, validate_weights, normalize_weights

# Test with old use cases
for uc in ['retail', 'ev_charging', 'warehouse', 'telecom', 'renewable']:
    w = get_default_weights(uc)
    wd = w.model_dump()
    is_valid, err = validate_weights(wd)
    status = '✅' if is_valid else '❌'
    print(f"  {status} {uc}: valid={is_valid}")

# Test with new use cases
for uc in ['hospital', 'liquor_store', 'data_center', 'solar_plant', 'bank_branch']:
    w = get_default_weights(uc)
    wd = w.model_dump()
    is_valid, err = validate_weights(wd)
    status = '✅' if is_valid else '❌'
    print(f"  {status} {uc}: valid={is_valid}")

print("\n✅ Config tools backward-compatible")

  ✅ retail: valid=True
  ✅ ev_charging: valid=True
  ✅ warehouse: valid=True
  ✅ telecom: valid=True
  ✅ renewable: valid=True
  ✅ hospital: valid=True
  ✅ liquor_store: valid=True
  ✅ data_center: valid=True
  ✅ solar_plant: valid=True
  ✅ bank_branch: valid=True

✅ Config tools backward-compatible


## Test 10: Prompts

In [18]:
from llm.prompts import (
    CHAT_INTENT_PROMPT, CHAT_FOLLOW_UP_PROMPT,
    CHAT_DIRECT_ANSWER_PROMPT, VALIDATION_INSIGHT_ADDENDUM,
    ADVISORY_SYSTEM_PROMPT, INSIGHT_SYSTEM_PROMPT,
)

prompts = {
    'CHAT_INTENT_PROMPT': CHAT_INTENT_PROMPT,
    'CHAT_FOLLOW_UP_PROMPT': CHAT_FOLLOW_UP_PROMPT,
    'CHAT_DIRECT_ANSWER_PROMPT': CHAT_DIRECT_ANSWER_PROMPT,
    'VALIDATION_INSIGHT_ADDENDUM': VALIDATION_INSIGHT_ADDENDUM,
    'ADVISORY_SYSTEM_PROMPT': ADVISORY_SYSTEM_PROMPT,
    'INSIGHT_SYSTEM_PROMPT': INSIGHT_SYSTEM_PROMPT,
}

for name, prompt in prompts.items():
    print(f"  ✅ {name}: {len(prompt)} chars")

print(f"\n✅ All {len(prompts)} prompts loaded")

  ✅ CHAT_INTENT_PROMPT: 1245 chars
  ✅ CHAT_FOLLOW_UP_PROMPT: 428 chars
  ✅ CHAT_DIRECT_ANSWER_PROMPT: 446 chars
  ✅ VALIDATION_INSIGHT_ADDENDUM: 190 chars
  ✅ ADVISORY_SYSTEM_PROMPT: 758 chars
  ✅ INSIGHT_SYSTEM_PROMPT: 317 chars

✅ All 6 prompts loaded


## Summary

All offline tests should pass without a database or LLM connection.
Tests 6-8 require a live DB and LLM API key — uncomment to run.